# Setup

In [ ]:
from datetime import date
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

df = catalog.load('raw/openaire/researchproduct_dev#parquet')

In [ ]:
def _pick_load_dt(df: pd.DataFrame):
    # Si hay una sola fecha en el batch, usala; si hay varias, quedate con la más reciente;
    # si no hay, hoy.
    if 'load_datetime' not in df.columns or df['_load_datetime'].isna().all():
        return date.today()
    vals = df['_load_datetime'].dropna()
    if vals.nunique() == 1:
        return vals.iloc[0]
    return pd.to_datetime(vals).max().date()

In [ ]:
df

# Profiling

## Perfilado de `organizations` en ResearchProduct (OpenAIRE)

Según [graph.openaire.eu/docs/data-model/entities/research-product/#organizations](https://graph.openaire.eu/docs/data-model/entities/research-product/#organizations), el atributo `organizations` de cada *ResearchProduct* es una **lista** de organizaciones vinculadas al recurso.  
Cada entrada incluye:

- `id`: identificador interno de OpenAIRE.  
- `legalName`: nombre legal de la organización.  
- `acronym`: acrónimo (puede no estar presente).  
- `pids`: lista de identificadores persistentes (ej. ROR, FundRef).  

**Decisiones de normalización en este nodo**:
- Desanidar `organizations` para generar una fila por par `researchproduct_id`–`organization_id`.  
- Renombrar claves para evitar colisiones: `id` → `researchproduct_id` (producto) y `id` → `organization_id` (organización).  
- Separar `pids` en columnas `pid_scheme` y `pid_value`.  
- Deduplicar `df_organizations` para conservar una única fila por `organization_id`.  
- Incorporar `load_datetime` para trazabilidad temporal.  

**Supuestos y riesgos**:
- `acronym` puede faltar, no usarlo como clave.  
- Una organización puede tener múltiples `pids`; no asumir unicidad por esquema.  
- El `organization_id` es interno a OpenAIRE; para interoperar se recomienda priorizar `ROR` en `pids`.  

**Checks mínimos (smoke tests)**:
- `df_research_organization`: contiene `researchproduct_id`, `organization_id`, `load_datetime`.  
- `df_organizations`: una fila por `organization_id` con `legalName`, `acronym`, `load_datetime`.  
- `df_organization_pid`: combina `organization_id` con cada `pid_scheme`–`pid_value`.  
- Invariantes:  
  - Todos los `organization_id` en `df_organizations` aparecen en `df_research_organization`.  
  - Todos los `organization_id` en `df_organization_pid` aparecen en `df_organizations`.  
  - `df_organizations` no tiene duplicados en `organization_id`.  


Se seleccionan los atributos _id_ y _organizations_; _organizations_ se expande y luego se reindexa el DataFrame descartando la indexación previa (reset_index(drop=True)). Finalmente, se renombra el atributo _id_ incorporando el nombre de la entidad (_researchproduct_), con el fin de evitar conflictos con identificadores de otras entidades.

In [ ]:
df_research_organization = df[['id','organizations']].explode('organizations').reset_index(drop=True)
df_research_organization.rename(columns={'id':'researchproduct_id'}, inplace=True)

In [ ]:
df_research_organization

In [ ]:
df_organizations = pd.json_normalize(df_research_organization['organizations'])
df_organizations.rename(columns={'id':'organization_id'}, inplace=True)

In [ ]:
df_organizations

In [ ]:
df_research_organization = pd.concat(
    [df_research_organization['researchproduct_id'], df_organizations['organization_id']], 
    axis=1
)

In [ ]:
df_research_organization

In [ ]:
df_organization_pid = df_organizations.loc[:, ['organization_id', 'pids']].copy()
df_organizations.drop(columns=['pids'], inplace=True)
df_organization_pid.dropna(inplace=True)

In [ ]:
df_organization_pid

In [ ]:
df_organization_pid = df_organization_pid.explode('pids', ignore_index=True)
df_organization_pid.loc[:, ['organization_id', 'pids']]

In [ ]:
df_organization_pid

In [ ]:
df_pid = pd.json_normalize(df_organization_pid['pids'])
df_pid.rename(columns={'scheme':'pid_scheme','value':'pid_value'}, inplace=True)

In [ ]:
df_pid

In [ ]:
df_organization_pid.drop(columns=['pids'], inplace=True)
df_organization_pid = pd.concat([df_organization_pid, df_pid], axis=1)

In [ ]:
df_organization_pid

Para asegurar una fila única por organization_id, se eliminan registros duplicados según el atributo organization_id conservando la primera aparición (drop_duplicates(subset="organization_id", keep="first")) y, a continuación, se reindexa el DataFrame descartando la indexación previa (reset_index(drop=True)).

In [ ]:
df_organizations = (
    df_organizations
    .drop_duplicates(subset="organization_id", keep="first")
    .reset_index(drop=True)
)

In [ ]:
df_organizations

# Node
+ info en https://graph.openaire.eu/docs/data-model/entities/research-product

In [ ]:
def openaire_land_researchproduct_organizations(df: pd.DataFrame)-> pd.DataFrame:

    load_dt = _pick_load_dt(df)

    df_research_organization = df[['id','organizations']].explode('organizations').reset_index(drop=True)
    df_research_organization.rename(columns={'id':'researchproduct_id'}, inplace=True)

    df_organizations = pd.json_normalize(df_research_organization['organizations'])
    df_organizations.rename(columns={'id':'organization_id'}, inplace=True)

    df_research_organization = pd.concat(
        [df_research_organization['researchproduct_id'], df_organizations['organization_id']], 
        axis=1
    )

    df_organization_pid = df_organizations.loc[:, ['organization_id', 'pids']].copy()
    df_organizations.drop(columns=['pids'], inplace=True)
    df_organization_pid.dropna(inplace=True)

    df_organization_pid = df_organization_pid.explode('pids', ignore_index=True)
    df_organization_pid.loc[:, ['organization_id', 'pids']]

    df_pid = pd.json_normalize(df_organization_pid['pids'])
    df_pid.rename(columns={'scheme':'pid_scheme','value':'pid_value'}, inplace=True)

    df_organization_pid.drop(columns=['pids'], inplace=True)
    df_organization_pid = pd.concat([df_organization_pid, df_pid], axis=1)

    df_organizations = (
        df_organizations
        .drop_duplicates(subset="organization_id", keep="first")
        .reset_index(drop=True)
    )
    
    df_organizations['_load_datetime'] = date.today()
    df_research_organization['_load_datetime'] = date.today()
    df_organization_pid['_load_datetime'] = date.today()

    return df_organizations, df_research_organization, df_organization_pid


# Smoke test

In [ ]:
df_organizations, df_researchproduct_organization, df_organization_pid = openaire_land_researchproduct_organizations(df)

In [ ]:
df_organizations

In [ ]:
df_researchproduct_organization

In [ ]:
df_organization_pid